# Cruce de archivos de Excel

Notebook para **cruzar dos archivos** (tipo `BUSCARV` / `VLOOKUP`, pero sin las limitaciones de Excel).

El flujo es siempre el mismo y va **bloque por bloque**:

| Bloque | Qué hace | ¿Tocas algo? |
|---|---|---|
| 1 | Librerías | No |
| 2 | Rutas de entrada y salida | **Sí** |
| 3 | Motor: lectura de archivos | No |
| 4 | Motor: normalización de llaves | No |
| 5 | Motor: cruce, auditoría y exportación | No |
| 6 | Lista los archivos disponibles | No |
| 7 | **Eliges los 2 archivos** (por índice) | **Sí** |
| 8 | **Eliges las columnas llave** (por índice) + diagnóstico | **Sí** |
| 9 | Fijas la normalización y se construyen las llaves | **Sí** |
| 10 | **Eliges qué columnas van al resultado** (por índice) | **Sí** |
| 11 | **Eliges el tipo de cruce** y se ejecuta | **Sí** |
| 12 | Auditoría de lo que no cruzó | No |
| 13 | Exporta el Excel final | No |
| 14 | (Opcional) Genera datos de ejemplo para probar | — |

**Convención:** el archivo **IZQUIERDO** es tu archivo base (el que quieres conservar completo) y el
**DERECHO** es del que traes información adicional.

> Si es la primera vez que lo corres, ve al **BLOQUE 14**, genera los datos de ejemplo y prueba el
> flujo completo antes de usar tus archivos reales.

---
## BLOQUE 1 — Librerías

Solo se necesitan `pandas` (manejo de tablas) y `openpyxl` (leer/escribir `.xlsx`).
`xlrd` únicamente si algún archivo viene en el formato viejo `.xls`.

Si falta alguna, descomenta la línea del `pip install`.

In [ ]:
# !pip install pandas openpyxl xlrd

from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("pandas", pd.__version__, "- listo")

---
## BLOQUE 2 — Configuración de rutas  ✏️

- `CARPETA_DATOS`: carpeta donde están los archivos que vas a cruzar.
- `CARPETA_SALIDA`: carpeta donde se guardará el Excel resultante.
- `NOMBRE_SALIDA`: nombre del archivo final.

Usa `r"..."` (string *raw*) para que las barras invertidas de Windows no den problema.

In [ ]:
CARPETA_DATOS = r"./datos"
CARPETA_SALIDA = r"./salida"
NOMBRE_SALIDA = "cruce_resultado.xlsx"

# Ejemplo en Windows:
# CARPETA_DATOS  = r"C:\Users\usuario\OneDrive\Escritorio\Cruces\Entrada"
# CARPETA_SALIDA = r"C:\Users\usuario\OneDrive\Escritorio\Cruces\Salida"

Path(CARPETA_DATOS).mkdir(parents=True, exist_ok=True)
Path(CARPETA_SALIDA).mkdir(parents=True, exist_ok=True)

print("Entrada:", Path(CARPETA_DATOS).resolve())
print("Salida :", Path(CARPETA_SALIDA).resolve())

---
## BLOQUE 3 — Motor: lectura de archivos

Aquí está el detalle **más importante de todo el notebook**:

> Todo se lee como **texto** (`dtype=str`).

¿Por qué? Porque si pandas lee un identificador como número:

- `0012345` se convierte en `12345` → **pierdes los ceros a la izquierda**
- `900123456` se convierte en `900123456.0` → **deja de cruzar con el texto `900123456`**

La función `leer_tabla` además:
- soporta `.xlsx`, `.xlsm`, `.xls` y `.csv` / `.tsv`;
- limpia el `.0` que Excel deja pegado a los enteros (`_arreglar_decimal`);
- reemplaza celdas vacías por `""` en vez de `NaN`, y quita espacios sobrantes.

`mostrar_columnas` imprime las columnas **numeradas** — esos números son los índices que usarás
en los bloques 8 y 10.

In [ ]:
EXTENSIONES = {".xlsx", ".xlsm", ".xls", ".csv", ".tsv"}
LLAVE = "__llave__"

_RE_DECIMAL = re.compile(r"^\s*(-?\d+)\.0+\s*$")


def _arreglar_decimal(valor: str) -> str:
    """'12345.0' -> '12345'. Deshace el decimal que Excel/pandas pegan a los enteros."""
    coincidencia = _RE_DECIMAL.match(valor)
    return coincidencia.group(1) if coincidencia else valor


def listar_archivos(carpeta) -> list[Path]:
    """Archivos cruzables de una carpeta. Ignora los temporales '~$' que deja Excel abierto."""
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        raise FileNotFoundError(f"La carpeta no existe: {carpeta}")
    return sorted(
        p for p in carpeta.iterdir()
        if p.suffix.lower() in EXTENSIONES and not p.name.startswith("~$")
    )


def hojas_de(ruta) -> list[str]:
    """Nombres de las hojas de un Excel (lista vacia para csv/tsv)."""
    ruta = Path(ruta)
    if ruta.suffix.lower() in (".csv", ".tsv"):
        return []
    motor = "xlrd" if ruta.suffix.lower() == ".xls" else "openpyxl"
    with pd.ExcelFile(ruta, engine=motor) as libro:
        return list(libro.sheet_names)


def leer_tabla(ruta, hoja=0) -> pd.DataFrame:
    """Lee el archivo COMPLETAMENTE como texto para no dañar los identificadores.

    hoja: indice (0 = primera) o nombre de la hoja. Solo aplica a Excel.
    """
    ruta = Path(ruta)
    sufijo = ruta.suffix.lower()

    if sufijo in (".csv", ".tsv"):
        separador = "\t" if sufijo == ".tsv" else None  # None => pandas detecta el separador
        df = pd.read_csv(ruta, dtype=str, sep=separador, engine="python", keep_default_na=False)
    else:
        motor = "xlrd" if sufijo == ".xls" else "openpyxl"
        try:
            df = pd.read_excel(ruta, sheet_name=hoja, dtype=str, engine=motor)
        except ImportError as exc:
            raise ImportError(
                f"Para leer archivos {sufijo} falta la libreria '{motor}'. "
                f"Instala con:  pip install {motor}"
            ) from exc

    df.columns = [str(c).strip() for c in df.columns]
    for columna in df.columns:
        df[columna] = df[columna].fillna("").astype(str).map(_arreglar_decimal).str.strip()
    return df.reset_index(drop=True)


def mostrar_columnas(df: pd.DataFrame, titulo: str) -> None:
    """Imprime las columnas NUMERADAS, con % de celdas llenas, valores unicos y ejemplos."""
    filas = []
    for i, columna in enumerate(df.columns):
        serie = df[columna]
        llenos = serie[serie != ""]
        filas.append({
            "idx": i,
            "columna": columna[:38],
            "lleno": f"{100 * len(llenos) / max(len(df), 1):.0f}%",
            "unicos": llenos.nunique(),
            "ejemplos": " | ".join(llenos.head(3).tolist())[:52],
        })
    print(f"\n=== {titulo}  ({len(df)} filas x {len(df.columns)} columnas) ===")
    print(pd.DataFrame(filas).to_string(index=False))


def elegir_columnas(df: pd.DataFrame, indices) -> list[str]:
    """Traduce indices -> nombres de columna. Acepta la palabra 'todas'."""
    if isinstance(indices, str) and indices.lower() == "todas":
        return list(df.columns)
    if isinstance(indices, int):
        indices = [indices]
    nombres = []
    for i in indices:
        if not 0 <= i < len(df.columns):
            raise IndexError(f"El indice {i} no existe. Validos: 0 a {len(df.columns) - 1}")
        nombres.append(df.columns[i])
    return nombres


print("Funciones de lectura cargadas.")

---
## BLOQUE 4 — Motor: normalización de llaves

Dos identificadores que **son el mismo** casi nunca están escritos igual en dos archivos.
Antes de cruzar hay que llevarlos a una forma común. Estos son los 5 modos disponibles:

| Modo | Qué hace | Ejemplo |
|---|---|---|
| `texto` | Quita tildes, pasa a MAYÚSCULAS, colapsa espacios | `" Gámma  S.A. "` → `GAMMA S.A.` |
| `nit` | Deja **solo dígitos** (quita puntos, guiones, espacios) | `900.123.456-7` → `9001234567` |
| `nit_sin_dv` | Solo dígitos y **quita el dígito de verificación** (el último) | `900.123.456-7` → `900123456` |
| `codigo` | Sin tildes, MAYÚSCULAS, **sin ningún espacio**. Conserva los ceros a la izquierda | `"AB 001 "` → `AB001` |
| `numerico` | Solo dígitos y **quita los ceros a la izquierda** | `0012345` → `12345` |

Y una opción adicional, `igualar_ceros`: si ambos lados quedan como puros dígitos, los rellena con
ceros a la izquierda hasta la misma longitud (`12345` y `0012345` → ambos `0012345`). Sirve cuando
un archivo conservó los ceros y el otro no.

### El bloque clave: `diagnosticar_llave`

En vez de que adivines qué modo usar, esta función **prueba las 50 combinaciones posibles y te dice
cuál cruza más**. Devuelve:

- `%_izq_encontrado`: de las llaves del archivo izquierdo, qué porcentaje aparece en el derecho.
- `%_der_encontrado`: lo mismo al revés.
- `asi_queda_la_llave`: cómo queda la llave ya normalizada (o `A != B` si no cruzó nada).

Las combinaciones que producen un resultado **idéntico** se colapsan en una sola fila, para que la
tabla no se llene de opciones repetidas.

In [ ]:
MODOS = ["texto", "nit", "nit_sin_dv", "codigo", "numerico"]
_RANGO_MODO = {"texto": 0, "nit": 1, "codigo": 2, "numerico": 3, "nit_sin_dv": 4}


def _sin_tildes(texto: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))


def normalizar_serie(serie: pd.Series, modo: str) -> pd.Series:
    """Aplica UNO de los 5 modos de normalizacion a una columna."""
    if modo not in MODOS:
        raise ValueError(f"Modo desconocido '{modo}'. Validos: {MODOS}")
    s = serie.fillna("").astype(str).map(_arreglar_decimal).str.strip()

    if modo == "texto":
        s = s.map(_sin_tildes).str.upper()
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    elif modo == "nit":
        s = s.str.replace(r"\D", "", regex=True)
    elif modo == "nit_sin_dv":
        s = s.str.replace(r"\D", "", regex=True).str[:-1]
    elif modo == "codigo":
        s = s.map(_sin_tildes).str.upper().str.replace(r"\s+", "", regex=True)
    elif modo == "numerico":
        s = s.str.replace(r"\D", "", regex=True).str.lstrip("0")
    return s


def igualar_ceros(sa: pd.Series, sb: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Si ambos lados son puros digitos, los rellena con ceros hasta la misma longitud."""
    juntos = pd.concat([sa[sa != ""], sb[sb != ""]])
    if juntos.empty or not juntos.str.fullmatch(r"\d+").all():
        return sa, sb  # hay letras o simbolos: rellenar con ceros no tendria sentido
    ancho = int(juntos.str.len().max())
    return (sa.str.zfill(ancho).where(sa != "", ""),
            sb.str.zfill(ancho).where(sb != "", ""))


def _cobertura(sa: pd.Series, sb: pd.Series) -> tuple[float, float]:
    """% de llaves NO vacias de cada lado que existen en el otro lado."""
    a, b = sa[sa != ""], sb[sb != ""]
    if a.empty or b.empty:
        return 0.0, 0.0
    return 100 * a.isin(set(b)).mean(), 100 * b.isin(set(a)).mean()


def _huella(sa: pd.Series, sb: pd.Series) -> int:
    """Firma del contenido ya normalizado, para colapsar combinaciones equivalentes."""
    return hash((tuple(sa.head(2000)), tuple(sb.head(2000))))


def diagnosticar_llave(sa: pd.Series, sb: pd.Series, top: int = 6) -> pd.DataFrame:
    """Prueba TODAS las combinaciones de modos y las ordena por cuanto cruzan."""
    filas = []
    for modo_a in MODOS:
        na = normalizar_serie(sa, modo_a)
        for modo_b in MODOS:
            nb = normalizar_serie(sb, modo_b)
            for pad in (False, True):
                xa, xb = igualar_ceros(na, nb) if pad else (na, nb)
                cob_a, cob_b = _cobertura(xa, xb)
                cruzados = xa[(xa != "") & xa.isin(set(xb))]
                filas.append({
                    "modo_izq": modo_a,
                    "modo_der": modo_b,
                    "igualar_ceros": pad,
                    "%_izq_encontrado": round(cob_a, 1),
                    "%_der_encontrado": round(cob_b, 1),
                    "asi_queda_la_llave": (
                        cruzados.iloc[0] if len(cruzados) else f"{xa.iloc[0]} != {xb.iloc[0]}"
                    )[:28],
                    "_huella": _huella(xa, xb),
                    "_rango": _RANGO_MODO[modo_a] + _RANGO_MODO[modo_b],
                })

    tabla = pd.DataFrame(filas).sort_values(
        ["%_izq_encontrado", "%_der_encontrado", "_rango", "igualar_ceros"],
        ascending=[False, False, True, True],
    )
    tabla = tabla.drop_duplicates(subset=["_huella"], keep="first")  # colapsa equivalentes
    return tabla.drop(columns=["_huella", "_rango"]).head(top).reset_index(drop=True)


def construir_llaves(df_a, cols_a, modos_a, df_b, cols_b, modos_b, igualar=True):
    """Construye la llave final de cada lado. Si hay varias columnas, las une con '||'."""
    if not (len(cols_a) == len(cols_b) == len(modos_a) == len(modos_b)):
        raise ValueError(
            "Las llaves izquierda y derecha deben tener la MISMA cantidad de columnas, "
            "y debe haber un modo por cada columna."
        )
    llave_a = llave_b = None
    for col_a, modo_a, col_b, modo_b in zip(cols_a, modos_a, cols_b, modos_b):
        na = normalizar_serie(df_a[col_a], modo_a)
        nb = normalizar_serie(df_b[col_b], modo_b)
        if igualar:
            na, nb = igualar_ceros(na, nb)
        llave_a = na if llave_a is None else llave_a + "||" + na
        llave_b = nb if llave_b is None else llave_b + "||" + nb
    return llave_a, llave_b


print("Funciones de normalizacion cargadas. Modos:", MODOS)

---
## BLOQUE 5 — Motor: cruce, auditoría y exportación

Tres cosas que este bloque resuelve y que en Excel te muerden en silencio:

1. **Llaves vacías.** Si dos filas tienen la llave en blanco, `merge` las considera iguales y las
   cruza. `_blindar_vacios` le da a cada vacío un valor único e irrepetible, así **nunca** cruzan.
2. **Nombres de columna repetidos.** Si ambos archivos tienen `Ciudad`, la del archivo derecho se
   renombra a `Ciudad_der` en vez de sobrescribirse.
3. **Llaves duplicadas.** Si una llave aparece 2 veces en el archivo derecho, esa fila del izquierdo
   se **duplica** en el resultado. Es el comportamiento correcto de un cruce, pero hay que verlo:
   por eso la hoja `llaves_duplicadas_der`.

La columna `_origen_` del resultado te dice, fila por fila, si `cruzo`, si quedó
`solo_izquierda` o `solo_derecha`.

In [ ]:
def _blindar_vacios(llave: pd.Series, prefijo: str) -> pd.Series:
    """Da a cada llave vacia un valor unico, para que los vacios NO crucen entre si."""
    relleno = pd.Series([f"__{prefijo}_VACIO_{i}__" for i in range(len(llave))], index=llave.index)
    return llave.where(llave != "", relleno)


def ejecutar_cruce(df_a, df_b, llave_a, llave_b, cols_izq, cols_der, tipo="left"):
    """Cruza los dos archivos.

    tipo:
      'left'  -> todas las filas del IZQUIERDO + lo que se encuentre del derecho (tipo BUSCARV)
      'inner' -> solo las filas que cruzan en AMBOS
      'outer' -> todo de ambos lados, cruce o no cruce
      'right' -> todas las filas del DERECHO + lo que se encuentre del izquierdo
    """
    if tipo not in ("left", "inner", "outer", "right"):
        raise ValueError("tipo debe ser: left, inner, outer o right")

    izq = df_a[list(cols_izq)].copy()
    izq[LLAVE] = _blindar_vacios(llave_a, "IZQ")

    der = df_b[list(cols_der)].copy()
    der[LLAVE] = _blindar_vacios(llave_b, "DER")

    choques = set(cols_izq) & set(cols_der)
    if choques:
        der = der.rename(columns={c: f"{c}_der" for c in choques})
        print(f"Nombres repetidos renombrados con sufijo '_der': {sorted(choques)}")

    resultado = izq.merge(der, on=LLAVE, how=tipo, indicator="_origen_")
    origen = resultado["_origen_"].astype(str).map(
        {"both": "cruzo", "left_only": "solo_izquierda", "right_only": "solo_derecha"}
    )
    resultado = resultado.drop(columns=[LLAVE, "_origen_"]).fillna("")
    resultado["_origen_"] = origen
    return resultado


def auditar(df_a, df_b, llave_a, llave_b, resultado):
    """Devuelve las hojas de auditoria y la tabla resumen."""
    set_a, set_b = set(llave_a[llave_a != ""]), set(llave_b[llave_b != ""])
    dup_der = llave_b[(llave_b != "") & llave_b.duplicated(keep=False)]

    hojas = {
        "sin_cruce_izq": df_a[~llave_a.isin(set_b) | (llave_a == "")].copy(),
        "sin_cruce_der": df_b[~llave_b.isin(set_a) | (llave_b == "")].copy(),
        "llaves_duplicadas_der": (
            df_b.loc[dup_der.index].assign(_llave_=dup_der).sort_values("_llave_")
        ),
    }
    resumen = pd.DataFrame([
        ("Filas archivo IZQUIERDO", len(df_a)),
        ("Filas archivo DERECHO", len(df_b)),
        ("Filas del RESULTADO", len(resultado)),
        ("Llaves unicas izquierda", len(set_a)),
        ("Llaves unicas derecha", len(set_b)),
        ("Llaves en comun", len(set_a & set_b)),
        ("Filas izq SIN cruce", len(hojas["sin_cruce_izq"])),
        ("Filas der SIN cruce", len(hojas["sin_cruce_der"])),
        ("Filas der con llave DUPLICADA", len(dup_der)),
        ("Llaves VACIAS izquierda", int((llave_a == "").sum())),
        ("Llaves VACIAS derecha", int((llave_b == "").sum())),
    ], columns=["concepto", "valor"])
    return hojas, resumen


def exportar(ruta_salida, resultado, hojas, resumen):
    """Escribe un unico Excel con el resultado y todas las hojas de auditoria."""
    ruta_salida = Path(ruta_salida)
    ruta_salida.parent.mkdir(parents=True, exist_ok=True)
    if len(resultado) > 1_048_575:
        raise ValueError(f"El resultado tiene {len(resultado):,} filas y no cabe en una hoja de Excel.")
    with pd.ExcelWriter(ruta_salida, engine="openpyxl") as libro:
        resumen.to_excel(libro, sheet_name="resumen", index=False)
        resultado.to_excel(libro, sheet_name="resultado", index=False)
        for nombre, hoja in hojas.items():
            hoja.to_excel(libro, sheet_name=nombre[:31], index=False)
    return ruta_salida


print("Motor de cruce cargado.")

---
## BLOQUE 6 — ¿Qué archivos hay disponibles?

Lista, **numerados**, los archivos cruzables que hay en `CARPETA_DATOS`.
Anota los dos números que te interesan: los vas a usar en el bloque siguiente.

In [ ]:
ARCHIVOS = listar_archivos(CARPETA_DATOS)

if not ARCHIVOS:
    print(f"No hay archivos cruzables en: {Path(CARPETA_DATOS).resolve()}")
    print("Copia ahi tus archivos, o corre el BLOQUE 14 para generar datos de ejemplo.")
else:
    print(f"Archivos en {Path(CARPETA_DATOS).resolve()}\n")
    for i, ruta in enumerate(ARCHIVOS):
        hojas = hojas_de(ruta)
        detalle = f"  hojas: {hojas}" if len(hojas) > 1 else ""
        print(f"  [{i}]  {ruta.name}  ({ruta.stat().st_size / 1024:,.0f} KB){detalle}")

---
## BLOQUE 7 — Elegir los dos archivos  ✏️

Escribe el **índice** de cada archivo según la lista del bloque anterior.

- `IDX_IZQUIERDO`: tu archivo base, el que quieres conservar completo.
- `IDX_DERECHO`: el archivo del que vas a traer información.

`HOJA_IZQ` / `HOJA_DER` solo importan si el Excel tiene varias hojas: puedes poner el número
(`0` = primera) o el nombre exacto de la hoja.

Al ejecutar verás las columnas de ambos archivos **numeradas**.

In [ ]:
IDX_IZQUIERDO = 0
IDX_DERECHO = 1

HOJA_IZQ = 0
HOJA_DER = 0

# ---------------------------------------------------------------------------
RUTA_IZQ, RUTA_DER = ARCHIVOS[IDX_IZQUIERDO], ARCHIVOS[IDX_DERECHO]
if RUTA_IZQ == RUTA_DER:
    print("AVISO: elegiste el mismo archivo en ambos lados.\n")

DF_IZQ = leer_tabla(RUTA_IZQ, HOJA_IZQ)
DF_DER = leer_tabla(RUTA_DER, HOJA_DER)

print(f"IZQUIERDO: {RUTA_IZQ.name}")
print(f"DERECHO  : {RUTA_DER.name}")

mostrar_columnas(DF_IZQ, f"IZQUIERDO - {RUTA_IZQ.name}")
mostrar_columnas(DF_DER, f"DERECHO - {RUTA_DER.name}")

---
## BLOQUE 8 — Elegir las columnas llave y diagnosticarlas  ✏️

Escribe los **índices** de las columnas por las que quieres cruzar, según la tabla del bloque anterior.

- Llave simple: `LLAVES_IZQ = [0]` y `LLAVES_DER = [0]`
- Llave compuesta: `LLAVES_IZQ = [0, 3]` y `LLAVES_DER = [0, 4]`

> Ambas listas deben tener la **misma cantidad** de columnas, y el **orden importa**: la primera de
> la izquierda se compara contra la primera de la derecha, y así sucesivamente.

Al ejecutar, verás para **cada pareja de llaves** las mejores combinaciones de normalización
ordenadas por cuánto cruzan. Mira la columna `%_izq_encontrado` y quédate con la fila de arriba:
esos son los modos que vas a copiar al bloque 9.

In [ ]:
LLAVES_IZQ = [0]
LLAVES_DER = [0]

# ---------------------------------------------------------------------------
COLS_LLAVE_IZQ = elegir_columnas(DF_IZQ, LLAVES_IZQ)
COLS_LLAVE_DER = elegir_columnas(DF_DER, LLAVES_DER)

if len(COLS_LLAVE_IZQ) != len(COLS_LLAVE_DER):
    raise ValueError(
        f"Cantidades distintas: izquierda {COLS_LLAVE_IZQ} vs derecha {COLS_LLAVE_DER}"
    )

for posicion, (col_i, col_d) in enumerate(zip(COLS_LLAVE_IZQ, COLS_LLAVE_DER)):
    print(f"\n{'=' * 78}")
    print(f"LLAVE {posicion + 1}:   izq '{col_i}'   <-->   der '{col_d}'")
    print("=" * 78)
    print(diagnosticar_llave(DF_IZQ[col_i], DF_DER[col_d]).to_string(index=False))

---
## BLOQUE 9 — Fijar la normalización y construir las llaves  ✏️

Copia aquí los modos que ganaron en el diagnóstico. Debe haber **un modo por cada columna llave**,
en el mismo orden.

```python
MODOS_IZQ = ["nit_sin_dv"]        # llave simple
MODOS_IZQ = ["nit_sin_dv", "texto"]  # llave compuesta de 2 columnas
```

Al ejecutar verás la cobertura real y, si algo no cruzó, **ejemplos concretos** de llaves que se
quedaron por fuera. Revísalos: casi siempre revelan el problema de fondo (una sede escrita distinto,
un NIT con letra, una fila de totales al final del archivo).

In [ ]:
MODOS_IZQ = ["nit_sin_dv"]   # <- valores del ejemplo del BLOQUE 14
MODOS_DER = ["nit"]
IGUALAR_CEROS = True

# ---------------------------------------------------------------------------
LLAVE_IZQ, LLAVE_DER = construir_llaves(
    DF_IZQ, COLS_LLAVE_IZQ, MODOS_IZQ,
    DF_DER, COLS_LLAVE_DER, MODOS_DER,
    igualar=IGUALAR_CEROS,
)

cob_izq, cob_der = _cobertura(LLAVE_IZQ, LLAVE_DER)
print(f"Cobertura izquierda->derecha : {cob_izq:6.1f}%")
print(f"Cobertura derecha->izquierda : {cob_der:6.1f}%")

print("\nAsi quedaron las llaves (primeras 5 filas):")
print(pd.DataFrame({"llave_izq": LLAVE_IZQ.head(5), "llave_der": LLAVE_DER.head(5)}).to_string(index=False))

huerfanas = LLAVE_IZQ[(LLAVE_IZQ != "") & ~LLAVE_IZQ.isin(set(LLAVE_DER))]
if len(huerfanas):
    print(f"\nOJO: {len(huerfanas)} llaves del archivo IZQUIERDO no existen en el derecho. Ejemplos:")
    for indice, valor in huerfanas.head(8).items():
        original = " | ".join(DF_IZQ.loc[indice, COLS_LLAVE_IZQ].astype(str))
        print(f"   fila {indice:>6}   llave '{valor}'   <- original: {original}")
else:
    print("\nTodas las llaves del archivo izquierdo cruzan.")

---
## BLOQUE 10 — Elegir qué columnas van al resultado  ✏️

Por **índice**, igual que antes. Puedes usar la palabra `"todas"`.

- `COLS_RESULTADO_IZQ`: columnas del archivo izquierdo que quieres conservar.
- `COLS_RESULTADO_DER`: columnas del archivo derecho que quieres **traer**.

> No hace falta incluir la columna llave del lado derecho: normalmente ya la tienes en el izquierdo.
> Si un nombre existe en ambos lados, el del derecho llegará con el sufijo `_der`.

Vuelve a correr el bloque 7 si necesitas ver la numeración de las columnas otra vez.

In [ ]:
COLS_RESULTADO_IZQ = "todas"
COLS_RESULTADO_DER = [2, 3]

# ---------------------------------------------------------------------------
NOMBRES_IZQ = elegir_columnas(DF_IZQ, COLS_RESULTADO_IZQ)
NOMBRES_DER = elegir_columnas(DF_DER, COLS_RESULTADO_DER)

print(f"Del IZQUIERDO se conservan {len(NOMBRES_IZQ)} columnas:")
print("   ", NOMBRES_IZQ)
print(f"\nDel DERECHO se traen {len(NOMBRES_DER)} columnas:")
print("   ", NOMBRES_DER)

---
## BLOQUE 11 — Tipo de cruce y ejecución  ✏️

| `TIPO_CRUCE` | Qué devuelve | Cuándo usarlo |
|---|---|---|
| `"left"` | Todas las filas del izquierdo + lo que encuentre del derecho | Lo más común: enriquecer tu base sin perder ninguna fila |
| `"inner"` | Solo las filas que cruzan en ambos | Cuando solo te sirve lo que efectivamente coincidió |
| `"outer"` | Todo de ambos lados, cruce o no | Para conciliar dos fuentes y ver todo el universo |
| `"right"` | Todas las filas del derecho + lo que encuentre del izquierdo | Poco frecuente; equivale a invertir los archivos |

In [ ]:
TIPO_CRUCE = "left"

# ---------------------------------------------------------------------------
RESULTADO = ejecutar_cruce(
    DF_IZQ, DF_DER, LLAVE_IZQ, LLAVE_DER,
    NOMBRES_IZQ, NOMBRES_DER, tipo=TIPO_CRUCE,
)

print(f"\nCruce '{TIPO_CRUCE}' -> {len(RESULTADO):,} filas x {len(RESULTADO.columns)} columnas\n")
print(RESULTADO["_origen_"].value_counts().to_string())

if len(RESULTADO) > len(DF_IZQ) and TIPO_CRUCE == "left":
    print(
        f"\nAVISO: el resultado tiene {len(RESULTADO) - len(DF_IZQ):,} filas MAS que el archivo "
        f"izquierdo.\nEsto pasa porque hay llaves repetidas en el archivo derecho "
        f"(revisa la hoja 'llaves_duplicadas_der')."
    )

print("\nPrimeras filas:")
RESULTADO.head(10)

---
## BLOQUE 12 — Auditoría

Antes de dar el cruce por bueno, revisa estos números. La pregunta que siempre hay que responder es
**"¿y lo que no cruzó, por qué no cruzó?"**.

- `sin_cruce_izq` → filas de tu base que se quedaron sin información.
- `sin_cruce_der` → filas del otro archivo que nunca se usaron.
- `llaves_duplicadas_der` → la causa de que el resultado tenga más filas de las esperadas.

In [ ]:
HOJAS, RESUMEN = auditar(DF_IZQ, DF_DER, LLAVE_IZQ, LLAVE_DER, RESULTADO)

print(RESUMEN.to_string(index=False))

for nombre, hoja in HOJAS.items():
    print(f"\n--- {nombre}: {len(hoja)} filas ---")
    if len(hoja):
        print(hoja.head(5).to_string(index=False))

---
## BLOQUE 13 — Exportar

Genera **un solo Excel** en `CARPETA_SALIDA` con estas hojas:

| Hoja | Contenido |
|---|---|
| `resumen` | Los números del bloque 12 |
| `resultado` | El cruce |
| `sin_cruce_izq` | Filas del izquierdo que no encontraron pareja |
| `sin_cruce_der` | Filas del derecho que nunca se usaron |
| `llaves_duplicadas_der` | Llaves repetidas en el derecho |

Como todo se manejó como texto, los **ceros a la izquierda se conservan** en el archivo final.

In [ ]:
RUTA_FINAL = exportar(Path(CARPETA_SALIDA) / NOMBRE_SALIDA, RESULTADO, HOJAS, RESUMEN)

print(f"Archivo generado: {RUTA_FINAL.resolve()}")
print(f"Tamano: {RUTA_FINAL.stat().st_size / 1024:,.0f} KB")
print(f"\nHojas: resumen, resultado, {', '.join(HOJAS)}")

---
## BLOQUE 14 — (Opcional) Datos de ejemplo para probar

Crea dos archivos en `CARPETA_DATOS` con **exactamente los problemas reales** que este notebook
resuelve, para que puedas probar el flujo completo sin tocar tus archivos:

- NIT con puntos y dígito de verificación (`900.123.456-7`) contra NIT limpio (`900123456`)
- Código interno con ceros a la izquierda (`0012345`) contra el mismo sin ceros (`12345`)
- Ciudad con tildes, espacios dobles y mayúsculas inconsistentes
- Un NIT **repetido** en el archivo derecho (para ver la multiplicación de filas)
- Una fila con el NIT **vacío**

Después de correr este bloque, vuelve al **BLOQUE 6** y sigue el flujo con:
`IDX_IZQUIERDO = 0`, `IDX_DERECHO = 1`, `LLAVES_IZQ = [0]`, `LLAVES_DER = [0]`.

In [ ]:
_izq = pd.DataFrame({
    "NIT":          ["900.123.456-7", "830.001.338-7", "  901555777-1", "800999888-2", ""],
    "COD_INTERNO":  ["0012345", "0000078", "0045000", "0000001", "0009999"],
    "Razon Social": ["Alfa S.A.S.", " BETA  LTDA ", "Gamma S.A.", "Delta SAS", "Epsilon"],
    "Ciudad":       ["Bogota", "Medellin", "Cali", "Bogota", "Cali"],
})
_der = pd.DataFrame({
    "nit_empresa": [900123456, 830001338, 901555777, 700111222, 900123456],
    "codigo":      [12345, 78, 45000, 55, 12345],
    "Contacto":    ["Ana", "Luis", "Marta", "Pedro", "Ana (segundo registro)"],
    "Correo":      ["ana@alfa.com", "luis@beta.com", "marta@gamma.com", "pedro@x.com", "ana2@alfa.com"],
    "Ciudad":      ["BOGOTA", "MEDELLIN", "CALI", "TUNJA", "BOGOTA"],
})

_carpeta = Path(CARPETA_DATOS)
_izq.to_excel(_carpeta / "01_clientes.xlsx", index=False)
_der.to_excel(_carpeta / "02_contactos.xlsx", index=False)

print(f"Creados en {_carpeta.resolve()}:")
print("  01_clientes.xlsx  (archivo IZQUIERDO)")
print("  02_contactos.xlsx (archivo DERECHO)")
print("\nAhora vuelve al BLOQUE 6.")